In [ ]:
import nltk
nltk.download('gutenberg')
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import tree2conlltags
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import re
from nltk.corpus.reader import PlaintextCorpusReader
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('wordnet')

In [ ]:
import requests

dickensDictionary = {
  "our-mutual-friend": "https://www.gutenberg.org/cache/epub/883/pg883.txt",
  "great-expectations": "https://www.gutenberg.org/cache/epub/1400/pg1400.txt", 
  "bleak-house": "https://www.gutenberg.org/cache/epub/1023/pg1023.txt",
  "great-expectations": "https://www.gutenberg.org/cache/epub/1400/pg1400.txt",
  "hard-times": "https://www.gutenberg.org/cache/epub/786/pg786.txt",
  "christmas-carol": "https://www.gutenberg.org/cache/epub/46/pg46.txt",
  "david-copperfield": "https://www.gutenberg.org/cache/epub/766/pg766.txt",
  "tale-of-two-cities": "https://www.gutenberg.org/cache/epub/98/pg98.txt",
  "oliver-twist": "https://www.gutenberg.org/cache/epub/730/pg730.txt",
  "pickwick-papers": "https://www.gutenberg.org/cache/epub/580/pg580.txt",
  "nicholas-nickleby": "https://www.gutenberg.org/cache/epub/967/pg967.txt",
  "old-curiosity-shop": "https://www.gutenberg.org/cache/epub/700/pg700.txt",
  "martin-chuzzlewit": "https://www.gutenberg.org/cache/epub/968/pg968.txt",
  "dombey-and-son": "https://www.gutenberg.org/cache/epub/821/pg821.txt",
  "barnaby-rudge": "https://www.gutenberg.org/cache/epub/917/pg917.txt",
  "american-notes": "https://www.gutenberg.org/cache/epub/675/pg675.txt",
  "sketches-by-boz": "https://www.gutenberg.org/cache/epub/882/pg882.txt",
  "mudfog-papers": "https://www.gutenberg.org/cache/epub/882/pg882.txt",
  "mystery-of-edwin-drood": "https://www.gutenberg.org/cache/epub/564/pg564.txt",
}



In [ ]:
# Open a new file in write-binary mode and write the content of the response to it
for item in dickensDictionary:
  response = requests.get(dickensDictionary[item])
  with open('data/dickens/' + item + '.txt', 'wb') as file:
    file.write(response.content)

In [ ]:
## This is meant to be for part of speech tagging

# Tokenize the text into sentences, then words
tokens = [word_tokenize(sent) for sent in nltk.sent_tokenize(text)]
# Tag the tokens with their part of speech
pos_tokens = [nltk.pos_tag(tok) for tok in tokens]
# Chunk the tagged tokens into named entities
chunked_tokens = [nltk.ne_chunk(tok) for tok in pos_tokens]
# Convert the trees into IOB tags
iob_tokens = [tree2conlltags(tok) for tok in chunked_tokens]


In [ ]:
# generate list of words associated with poverty

# Specify the word you're interested in
words = ['poor', "needy", "indigent", "beggar"]

# Get the synsets for the word
rawsynsets = [wordnet.synsets(word) for word in words]
synsets = [item for sublist in rawsynsets for item in sublist]

# Get the lemmas for each synset and add them to a list
associated_words = [lemma.name() for synset in synsets for lemma in synset.lemmas()]

associated_words = list(set(associated_words))



In [ ]:
# Get corpus 

# Specify the directory of your corpus
corpus_dir = 'data/dickens/'  # Change this to the directory of your corpus

# Create a new corpus by creating a new PlaintextCorpusReader object
new_corpus = PlaintextCorpusReader(corpus_dir, '.*')


In [ ]:
# conFreq = nltk.ConditionalFreqDist((token, genre) 
#                                    for genre in new_corpus.fileids()
#                                    for token, pos in nltk.pos_tag(new_corpus.words(fileids=genre))
#                                    if pos == "JJ")

In [ ]:
# clean corpus
# lematize words
# remove punctuation
# making everything lower case

# Create a new WordNetLemmatizer object
lemmatizer = WordNetLemmatizer()

# Lemmatize all the words in your corpus
lemmatized_corpus = []
for id in new_corpus.fileids():
  lemmatized_corpus.append(" ".join([lemmatizer.lemmatize(word) for word in new_corpus.words(fileids=id)]))
  
lemmatized_corpus = [re.sub(r'[^\w\s]', '', doc) for doc in lemmatized_corpus]
lemmatized_corpus = [doc.lower() for doc in lemmatized_corpus]

In [ ]:
# tokenize the resulting cleaned text
tokenized_lemmatized_corpus = [word_tokenize(doc) for doc in lemmatized_corpus]

In [ ]:
conFreq = nltk.ConditionalFreqDist((token, genre) 
                                   for index,genre in enumerate(new_corpus.fileids())
                                   for token, pos in nltk.pos_tag(tokenized_lemmatized_corpus[index])
                                   if token in associated_words)

In [ ]:
## us this to create a sorted listed of tuples
sortedConFreqsDict = {} 

for index, keyToUse in enumerate(associated_words):
  sortedConFreqsDict[keyToUse] = sorted(conFreq[keyToUse].items(), key=lambda x: x[0], reverse=False)

  ## use this add null values to list
for fileid in new_corpus.fileids():
  for key in sortedConFreqsDict.keys():
    # test if file id is in any of the first value positions in any of the tuples in the list
    if fileid not in [tup[0] for tup in sortedConFreqsDict[key]]:
      sortedConFreqsDict[key].append((fileid, 0))

# sort again 

## us this to create a sorted listed of tuples
newSortedConFreqsDict = {} 

for index, keyToUse in enumerate(associated_words):
  newSortedConFreqsDict[keyToUse] = sorted(sortedConFreqsDict[keyToUse], key=lambda x: x[0], reverse=False)

In [ ]:
# # use this filter values
# filteredSortedConFreqsDict = {} 

# for index, keyToUse in enumerate(associated_words):
#   incomingList = [value for name, value in list(newSortedConFreqsDict[keyToUse])]
#   #if detect_stdv(incomingList) > 3: 
#   #if detect_shift_with_z_score(incomingList, threshold=8):
#   #if detect_unique(incomingList, uniqueMinThreshold=1, uniqueMaxThreshold=2):
#   filteredSortedConFreqsDict[keyToUse] = newSortedConFreqsDict[keyToUse]

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(30, 20),)  # Set the size of the figure
plt.title(f"Frequency Poverty attributes in Dickens novels")
plt.xticks(rotation=90)
for key in list(sortedConFreqsDict): 
    
    plt.plot([x[0] for x in sortedConFreqsDict[key]], [x[1] for x in sortedConFreqsDict[key]], label=key)

plt.legend(loc='upper right')    
plt.show()

In [ ]:
# create a new dictionary to sum up frequency for all values

newDict = {}
for text in sortedConFreqsDict[list(sortedConFreqsDict.keys())[0]]:
  newDict[text[0]] = 0 


In [ ]:
for keys in sortedConFreqsDict.keys():
  for index, text in enumerate(sortedConFreqsDict[keys]):
    newDict[text[0]] += text[1]

In [ ]:
for key in newDict.keys():
  newDict[key] = round((newDict[key] / len(new_corpus.words(fileids=key)) * 100), 2)
  

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(list(newDict.keys()), list(newDict.values()))
plt.xticks(rotation=90)
plt.title('Title')  # Replace with your title
plt.xlabel('X-axis label')  # Replace with your x-axis label
plt.ylabel('Y-axis label')  # Replace with your y-axis label
plt.show()


In [ ]:
# Assuming 'new_corpus' is your corpus
words = new_corpus.words()  # Get all the words in the corpus

# Create a nltk.Text object from the words
text = nltk.Text(words)

# Use the concordance method
text.concordance('poor', width=100)  # Replace 'your_word' with the word you're interested in

In [ ]:
def find_nearby_words(text, word1, word2, window):
    text = nltk.Text(text)
    word1_indexes = [i for i, word in enumerate(text) if word == word1]
    for index in word1_indexes:
        left = max(0, index - window)
        right = index + window + 1
        window_words = text[left:right]
        if word2 in window_words:
            print(f"'{word1}' and '{word2}' found within {window} words of each other.")
            print(" ".join(window_words))

# Assuming 'corpus' is your text
find_nearby_words(text, 'David', 'Eugene', 100)  # Replace 'word1' and 'word2' with your words